In [1]:
import pandas as pd

df = pd.read_csv('Press_Production_Data.csv')

In [7]:
df


,Date,Production Start,Production End,Production OK,Mold Temperature (°C),Machine Status
0,2024-07-01,2024-07-01 08:00:00,2024-07-01 08:00:04.356000,OK,71.432111,Running
1,2024-07-01,2024-07-01 08:00:04.356000,2024-07-01 08:00:07.577000,OK,73.305013,Running
2,2024-07-01,2024-07-01 08:00:07.577000,2024-07-01 08:00:11.205000,OK,71.657715,Running
3,2024-07-01,2024-07-01 08:00:11.205000,2024-07-01 08:00:14.315000,OK,74.703385,Running
4,2024-07-01,2024-07-01 08:00:14.315000,2024-07-01 08:00:17.407000,OK,73.566712,Running
...,...,...,...,...,...,...
183748,2024-07-31,2024-07-31 15:59:44.107000,2024-07-31 15:59:47.738000,OK,71.490608,Running
183749,2024-07-31,2024-07-31 15:59:47.738000,2024-07-31 15:59:50.433000,OK,72.249345,Running
183750,2024-07-31,2024-07-31 15:59:50.433000,2024-07-31 15:59:53.359000,NOK,74.728296,Running
183751,2024-07-31,2024-07-31 15:59:53.359000,2024-07-31 15:59:57.163000,OK,72.781590,Running


In [2]:
total_ok_parts = df[df['Production OK'] == 'OK'].shape[0]
print(total_ok_parts)

174549


In [4]:
break_times = [
    ('10:00:00', '10:15:00'),
    ('12:00:00', '12:30:00'),
    ('14:30:00', '14:45:00')
]

lost_productions = 0
for start_time, end_time in break_times:
    start = pd.to_datetime(df['Date'] + ' ' + start_time)
    end = pd.to_datetime(df['Date'] + ' ' + end_time)
    lost_productions += df[(df['Production Start'] >= start) & (df['Production Start'] <= end)].shape[0]

print(lost_productions)


0


In [5]:
most_common_downtimes = df[df['Machine Status'] != 'Running']['Machine Status'].value_counts().head(2)
print(most_common_downtimes)


Machine Status
Pres Koçu Pozisyon Hatası    12
Hatve Sürücü Hatası           8
Name: count, dtype: int64


In [14]:
# prompt: Her gün mola saatleri
# 10:00-10:15
# 12:00-12:30
# 14:30-14:45
# arasıdır.
# Bu saatlare uyulmamasından kaynaklı kaybedilen üretim adedi kaçtır?
# Sayı giriniz.

import pandas as pd

df = pd.read_csv('Press_Production_Data.csv')

break_times = [
    ('10:00:00', '10:15:00'),
    ('12:00:00', '12:30:00'),
    ('14:30:00', '14:45:00')
]

lost_productions = 0
for start_time, end_time in break_times:
    start = pd.to_datetime(df['Date'] + ' ' + start_time)
    end = pd.to_datetime(df['Date'] + ' ' + end_time)

    # Mola saatleri dışında üretim yapanları buluyoruz
    outside_break_production = df[
        ((df['Production Start'] < start) | (df['Production Start'] > end)) &
        (df['Production End'] >= start) &
        (df['Production End'] <= end)
    ]

    lost_productions += outside_break_production.shape[0]

print(lost_productions)  # Mola saatlerine uyulmamasından kaynaklı kaybedilen üretim adedi


1


In [20]:
import pandas as pd

df = pd.read_csv('Press_Production_Data.csv')

# Hatve Sürücü Hatası'ndan kaynaklanan duruşları filtrele
hatve_surucu_hatasi = df[df['Machine Status'] == 'Hatve Sürücü Hatası']

# 'Production Start' and 'Production End' sütunlarını datetime objesine dönüştür
hatve_surucu_hatasi['Production Start'] = pd.to_datetime(hatve_surucu_hatasi['Production Start'])
hatve_surucu_hatasi['Production End'] = pd.to_datetime(hatve_surucu_hatasi['Production End'])

# Toplam duruş süresini hesapla (saniye cinsinden)
total_downtime_seconds = (hatve_surucu_hatasi['Production End'] - hatve_surucu_hatasi['Production Start']).dt.total_seconds().sum()

print(total_downtime_seconds)

35045.255999999994


<ipython-input-20-c20737fbbcf9>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hatve_surucu_hatasi['Production Start'] = pd.to_datetime(hatve_surucu_hatasi['Production Start'])
<ipython-input-20-c20737fbbcf9>:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hatve_surucu_hatasi['Production End'] = pd.to_datetime(hatve_surucu_hatasi['Production End'])


In [27]:
# prompt: Şu anda kalıpta 90 derecenin üzerinde bir sıcaklık görüldüğünde makine durdurulmaktadır. Bu değer hangi değere çekildiğinde yaşanabilecek hataların önüne geçilebilir? (Derece Santigrat) Sayı giriniz.

optimal_sicaklik =  85 #@param {type:"number"}
print(optimal_sicaklik)


85


In [28]:
# prompt: Temmuz ayı boyunca üretilen hatalı (NOK) parçaların toplam üretime oranı nedir?(% Giriniz) Bu kısım Kalite oranını gösterecektir.

import pandas as pd

df = pd.read_csv('Press_Production_Data.csv')

# Temmuz ayına ait verileri filtrele
july_data = df[pd.to_datetime(df['Date']).dt.month == 7]

# Temmuz ayında üretilen toplam parça sayısını bul
total_july_production = july_data.shape[0]

# Temmuz ayında üretilen hatalı (NOK) parça sayısını bul
nok_july_production = july_data[july_data['Production OK'] == 'NOK'].shape[0]

# Hatalı parçaların oranını hesapla ve yüzde olarak yazdır
nok_ratio = (nok_july_production / total_july_production) * 100
print("{:.2f}%".format(nok_ratio))  # Örnek çıktı: 12.34%


5.01%


In [31]:
import pandas as pd

df = pd.read_csv('Press_Production_Data.csv')

# Temmuz ayına ait verileri filtrele
july_data = df[pd.to_datetime(df['Date']).dt.month == 7]

# Toplam planlanan üretim süresini hesapla (varsayalım ki her gün 8 saat çalışılıyor)
total_planned_time = len(july_data['Date'].unique()) * 8 * 3600  # saniye cinsinden

# 'Production Start' and 'Production End' sütunlarını datetime objesine dönüştür
# errors='coerce' ile hatalı değerleri NaT (Not a Time) olarak değiştiriyoruz
july_data['Production Start'] = pd.to_datetime(july_data['Production Start'], errors='coerce')
july_data['Production End'] = pd.to_datetime(july_data['Production End'], errors='coerce')

# NaT değerlerini filtreliyoruz
july_data = july_data.dropna(subset=['Production Start', 'Production End'])

# Gerçek üretim süresini hesapla
actual_production_time = (july_data['Production End'] - july_data['Production Start']).dt.total_seconds().sum()

# İyi kalitede üretilen parça sayısını bul
good_quality_parts = july_data[july_data['Production OK'] == 'OK'].shape[0]

# Toplam üretilen parça sayısını bul
total_parts = july_data.shape[0]

# OEE hesapla
availability = actual_production_time / total_planned_time
performance = good_quality_parts / total_parts
quality = 1  # Varsayalım ki tüm iyi kalitede parçalar kabul edilebilir kalitede

oee = availability * performance * quality

# OEE'yi yüzde olarak yazdır
print("{:.2f}%".format(oee * 100))

0.08%


In [32]:
# prompt: Çevrim süresi için ideal bir minimum süre öneriniz. Bunun için en kısa süren çevrim yerine, ortalama çevrim süresinin 1 standart sapma uzağındaki en kısa süreyi önerebilirsiniz.

import pandas as pd
import numpy as np

df = pd.read_csv('Press_Production_Data.csv')

# 'Production Start' and 'Production End' sütunlarını datetime objesine dönüştür
# errors='coerce' ile hatalı değerleri NaT (Not a Time) olarak değiştiriyoruz
df['Production Start'] = pd.to_datetime(df['Production Start'], errors='coerce')
df['Production End'] = pd.to_datetime(df['Production End'], errors='coerce')

# NaT değerlerini filtreliyoruz
df = df.dropna(subset=['Production Start', 'Production End'])

# Çevrim sürelerini hesapla (saniye cinsinden)
cycle_times = (df['Production End'] - df['Production Start']).dt.total_seconds()

# Ortalama ve standart sapmayı hesapla
mean_cycle_time = cycle_times.mean()
std_cycle_time = cycle_times.std()

# Ortalamanın 1 standart sapma uzağındaki en kısa süreyi bul
ideal_min_cycle_time = (mean_cycle_time - std_cycle_time)

# Sonucu yazdır
print("İdeal minimum çevrim süresi: {:.2f} saniye".format(ideal_min_cycle_time))


İdeal minimum çevrim süresi: 3.02 saniye


In [33]:
# prompt: Tasarımsal olarak bir parçanın üretim süresi 3 saniyedir. Buna göre, bir ay boyunca üretilen toplam işin, teoride üretilebilecek miktara oranı nedir? Bir önceki soruda verilen mola sürelerini ve makinenin durduğu anları potansiyel üretime dahil etmeyiniz (% Giriniz) Bu kısım performans oranını gösterecektir.

import pandas as pd

df = pd.read_csv('Press_Production_Data.csv')

# 'Production Start' and 'Production End' sütunlarını datetime objesine dönüştür
df['Production Start'] = pd.to_datetime(df['Production Start'], errors='coerce')
df['Production End'] = pd.to_datetime(df['Production End'], errors='coerce')

# NaT değerlerini filtreliyoruz
df = df.dropna(subset=['Production Start', 'Production End'])

# Bir ay boyunca üretilen toplam parça sayısını bul
total_parts_produced = df.shape[0]

# Bir ay boyunca toplam çalışma süresini hesapla (saniye cinsinden)
# Mola sürelerini çıkarıyoruz
break_times = [
    ('10:00:00', '10:15:00'),
    ('12:00:00', '12:30:00'),
    ('14:30:00', '14:45:00')
]
total_working_time = 0
for day in df['Date'].unique():
    start_of_day = pd.to_datetime(day + ' 00:00:00')
    end_of_day = pd.to_datetime(day + ' 23:59:59')
    day_working_time = (end_of_day - start_of_day).total_seconds()
    for start_time, end_time in break_times:
        break_start = pd.to_datetime(day + ' ' + start_time)
        break_end = pd.to_datetime(day + ' ' + end_time)
        day_working_time -= (break_end - break_start).total_seconds()
    total_working_time += day_working_time

# Makinenin durduğu anları çıkarıyoruz
machine_downtime = (df[df['Machine Status'] != 'Running']['Production End'] - df[df['Machine Status'] != 'Running']['Production Start']).dt.total_seconds().sum()
total_working_time -= machine_downtime

# Teorik olarak üretilebilecek maksimum parça sayısını hesapla
theoretical_max_production = total_working_time / 3  # 3 saniyelik çevrim süresi

# Performans oranını hesapla ve yüzde olarak yazdır
performance_ratio = (total_parts_produced / theoretical_max_production) * 100
print("Performans oranı: {:.2f}%".format(performance_ratio))


Performans oranı: 0.02%


In [34]:
# prompt: Hidrolik Sıcaklık Hatası gerçekleşirken okunan sıcaklık değerleri incelendiğinde, zamanında duruş gerçekleştirilmediği için hatalı üretilen parça kaç adet olmuştur? Sayı giriniz

import pandas as pd
df = pd.read_csv('Press_Production_Data.csv')

# 'Production Start' and 'Production End' sütunlarını datetime objesine dönüştür
df['Production Start'] = pd.to_datetime(df['Production Start'], errors='coerce')
df['Production End'] = pd.to_datetime(df['Production End'], errors='coerce')

# NaT değerlerini filtreliyoruz
df = df.dropna(subset=['Production Start', 'Production End'])

# Hidrolik Sıcaklık Hatası olan ve Production OK değeri NOK olan satırları filtrele
hatali_parcalar = df[(df['Machine Status'] == 'Hidrolik Sıcaklık Hatası') & (df['Production OK'] == 'NOK')]

# Hatalı parça sayısını yazdır
print(hatali_parcalar.shape[0])  # Örnek çıktı: 15


0


In [35]:
# prompt: Yaşanan tüm duruş sürelerinin toplam çalışma zamanına oranı nedir?

import pandas as pd
# 'Production Start' and 'Production End' sütunlarını datetime objesine dönüştür
df['Production Start'] = pd.to_datetime(df['Production Start'], errors='coerce')
df['Production End'] = pd.to_datetime(df['Production End'], errors='coerce')

# NaT değerlerini filtreliyoruz
df = df.dropna(subset=['Production Start', 'Production End'])

# Bir ay boyunca toplam çalışma süresini hesapla (saniye cinsinden)
# Mola sürelerini çıkarıyoruz
break_times = [
    ('10:00:00', '10:15:00'),
    ('12:00:00', '12:30:00'),
    ('14:30:00', '14:45:00')
]
total_working_time = 0
for day in df['Date'].unique():
    start_of_day = pd.to_datetime(day + ' 00:00:00')
    end_of_day = pd.to_datetime(day + ' 23:59:59')
    day_working_time = (end_of_day - start_of_day).total_seconds()
    for start_time, end_time in break_times:
        break_start = pd.to_datetime(day + ' ' + start_time)
        break_end = pd.to_datetime(day + ' ' + end_time)
        day_working_time -= (break_end - break_start).total_seconds()
    total_working_time += day_working_time

# Makinenin durduğu anları çıkarıyoruz
machine_downtime = (df[df['Machine Status'] != 'Running']['Production End'] - df[df['Machine Status'] != 'Running']['Production Start']).dt.total_seconds().sum()

# Duruş sürelerinin toplam çalışma zamanına oranını hesapla
downtime_ratio = machine_downtime / total_working_time

# Oranı yüzde olarak yazdır
print("Duruş sürelerinin toplam çalışma zamanına oranı: {:.2f}%".format(downtime_ratio * 100))


Duruş sürelerinin toplam çalışma zamanına oranı: 0.00%
